## Exploratory Data Analysis (EDA)

Here we will explore the shape of the data and the distribution of the labels.

In [16]:
# Import libraries and set up env
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

# Only these libraries are allowed for Project 1 (stdlib + NumPy;
# matplotlib/seaborn for visualization only -- no pandas / scikit-learn).
# import seaborn as sns

In [17]:
# First we load the data

# helpers.py lives in the `project1` folder (next to this notebook) and reads
# the three csv files from its `data` sub-folder. Locate that folder
# regardless of the folder the Jupyter kernel was started in.
_CANDIDATES = [
    os.getcwd(),                            # kernel started in project1/
    os.path.join(os.getcwd(), "project1"),  # kernel started at the repo root
    os.path.dirname(os.getcwd()),           # kernel started in a sub-folder
]
PROJECT_DIR = next(
    (p for p in _CANDIDATES if os.path.isfile(os.path.join(p, "helpers.py"))),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find project1/helpers.py -- check the kernel's working directory."
    )

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from helpers import load_csv_data

DATA_DIR = os.path.join(PROJECT_DIR, "data")
CACHE_DIR = os.path.join(DATA_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

cache_files = ["x_train.npy", "x_test.npy", "y_train.npy", "train_ids.npy", "test_ids.npy"]
all_cached = all(os.path.exists(os.path.join(CACHE_DIR, f)) for f in cache_files)

# We cache the data as .npy files to speed up the loading process.

if all_cached:
    print("Loading data from fast binary cache...")
    x_train = np.load(os.path.join(CACHE_DIR, "x_train.npy"))
    x_test = np.load(os.path.join(CACHE_DIR, "x_test.npy"))
    y_train = np.load(os.path.join(CACHE_DIR, "y_train.npy"))
    train_ids = np.load(os.path.join(CACHE_DIR, "train_ids.npy"))
    test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
else:
    print("Parsing CSVs via genfromtxt (this will take a while once), then caching...")
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_DIR)
    
    np.save(os.path.join(CACHE_DIR, "x_train.npy"), x_train)
    np.save(os.path.join(CACHE_DIR, "x_test.npy"), x_test)
    np.save(os.path.join(CACHE_DIR, "y_train.npy"), y_train)
    np.save(os.path.join(CACHE_DIR, "train_ids.npy"), train_ids)
    np.save(os.path.join(CACHE_DIR, "test_ids.npy"), test_ids)

print(f"DATA_DIR  : {DATA_DIR}")
print(f"x_train   : {x_train.shape}  (dtype={x_train.dtype})")
print(f"x_test    : {x_test.shape}")
print(f"y_train   : {y_train.shape}  values={np.unique(y_train)}")
print(f"train_ids : {train_ids.shape}")
print(f"test_ids  : {test_ids.shape}")

labels, counts = np.unique(y_train, return_counts=True)
for lab, cnt in zip(labels, counts):
    print(f"  label {int(lab):+d}: {cnt:>8d}  ({100 * cnt / y_train.size:5.2f}%)")

Loading data from fast binary cache...
DATA_DIR  : c:\Users\riri9\Desktop\MA1\ML\CS-433-project\project1\data
x_train   : (328135, 321)  (dtype=float64)
x_test    : (109379, 321)
y_train   : (328135,)  values=[-1  1]
train_ids : (328135,)
test_ids  : (109379,)
  label -1:   299160  (91.17%)
  label +1:    28975  ( 8.83%)


We can already see a pretty heavy class imbalance (less than 1/10 of the samples present heart attack symptoms).

We can also see that there appears to be a lot of features, but we can see from the [dataset codebook details](https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf) that not all of them are very relevant to predict a heart attack risk (for instance metadata such as "Interview date", "State FIPS code", "Number of adults in the household" etc.).
Therefore in the next step, we will handpick a few relevant features to reduce the dimensionality of the problem. To help us, we took notes from 2 specific work related to our dataset:
1. **Zidian Xie et al. (2019)**: *[Building Risk Prediction Models for Type 2 Diabetes Using Machine Learning Techniques](https://www.cdc.gov/pcd/issues/2019/19_0109.htm)* – A peer-reviewed study that evaluated the BRFSS dataset to identify core lifestyle and metabolic predictors. Because diabetes and cardiovascular disease share major risk factors, this research serves as a strong medical baseline for feature selection.
2. **Alex Teboul's Kaggle Analysis**: *[Heart Disease Health Indicators Dataset Notebook](https://www.kaggle.com/code/alexteboul/heart-disease-health-indicators-dataset-notebook)* – A data curation project that directly applied the findings from Xie et al. to the 2015 BRFSS dataset, successfully stripping away administrative noise to isolate the 22 most impactful variables (1 target, 21 independent) for predicting heart disease. As the author noted, these 21 features were adapted from the first study because Type 2 diabetes and cardiovascular disease share the exact same underlying metabolic and lifestyle risk factors. [(other source)](https://www.cdc.gov/diabetes/diabetes-complications/diabetes-and-your-heart.html).

In [18]:
# 21 relevant features from the research and notebook cited above
selected_features = [
    '_RFHYPE5', 'TOLDHI2', '_CHOLCHK', '_BMI5', 'SMOKE100', 'CVDSTRK3', 
    'DIABETE3', '_TOTINDA', '_FRTLT1', '_VEGLT1', '_RFDRHV5', 'HLTHPLN1', 
    'MEDCOST', 'GENHLTH', 'MENTHLTH', 'PHYSHLTH', 'DIFFWALK', 'SEX', 
    '_AGEG5YR', 'EDUCA', 'INCOME2'
]

In [19]:
def cleanse_feature(data_array, csv_path):
    """
    Filters a NumPy array to keep only the 21 selected features based on the CSV header.
    
    Args:
        data_array (np.array): The loaded dataset (e.g., x_train or x_test) without the ID column.
        csv_path (str): Path to the corresponding raw CSV file to read the column names.
        
    Returns:
        np.array: The curated dataset sliced to the 21 relevant features.
    """
    # Read just the first line of the CSV to get the header
    with open(csv_path, "r") as f:
        header_line = f.readline().strip()
        
    all_columns = [col.strip('"') for col in header_line.split(',')]
    
    # Calculate indices. Subtract 1 because load_csv_data already stripped the first column (Id)
    feature_indices = [all_columns.index(feat) - 1 for feat in selected_features]
    
    # Return the sliced array
    return data_array[:, feature_indices]


# Apply the function uniformly to both datasets, saving to new variables to prevent indexing errors
x_train_clean = cleanse_feature(x_train, os.path.join(DATA_DIR, "x_train.csv"))
x_test_clean = cleanse_feature(x_test, os.path.join(DATA_DIR, "x_test.csv"))

print(f"Curated x_train shape: {x_train_clean.shape}")
print(f"Curated x_test shape:  {x_test_clean.shape}")

Curated x_train shape: (328135, 21)
Curated x_test shape:  (109379, 21)


Now that we have more or less only what we need, we can checkout more in details what the features are actually made of.

In [20]:
print(f"{'Feature Name':<12} | {'Unique':<7} | {'Min':<5} | {'Max':<5} | {'Inferred Type'}")
print("-" * 75)

for i, feature_name in enumerate(selected_features):
    col_data = x_train_clean[:, i] 
    # Ignore NaNs if any exist to prevent calculation errors
    col_data = col_data[~np.isnan(col_data)]
    
    # Calculate statistics
    unique_vals = np.unique(col_data)
    num_unique = len(unique_vals)
    val_min = np.min(col_data)
    val_max = np.max(col_data)
    
    # Infer the data type based on the unique value count
    if num_unique == 2:
        inferred = f"Binary ({unique_vals[0]}, {unique_vals[1]})"
    elif num_unique < 20:
        inferred = f"Categorical/Ordinal"
    else:
        inferred = f"Continuous"
        
    print(f"{feature_name:<12} | {num_unique:<7} | {val_min:<5.0f} | {val_max:<5.0f} | {inferred}")

Feature Name | Unique  | Min   | Max   | Inferred Type
---------------------------------------------------------------------------
_RFHYPE5     | 3       | 1     | 9     | Categorical/Ordinal
TOLDHI2      | 4       | 1     | 9     | Categorical/Ordinal
_CHOLCHK     | 4       | 1     | 9     | Categorical/Ordinal
_BMI5        | 3524    | 12    | 98    | Continuous
SMOKE100     | 4       | 1     | 9     | Categorical/Ordinal
CVDSTRK3     | 4       | 1     | 9     | Categorical/Ordinal
DIABETE3     | 6       | 1     | 9     | Categorical/Ordinal
_TOTINDA     | 3       | 1     | 9     | Categorical/Ordinal
_FRTLT1      | 3       | 1     | 9     | Categorical/Ordinal
_VEGLT1      | 3       | 1     | 9     | Categorical/Ordinal
_RFDRHV5     | 3       | 1     | 9     | Categorical/Ordinal
HLTHPLN1     | 4       | 1     | 9     | Categorical/Ordinal
MEDCOST      | 4       | 1     | 9     | Categorical/Ordinal
GENHLTH      | 7       | 1     | 9     | Categorical/Ordinal
MENTHLTH     | 33      |

Now, we can further clean our dataset by dropping rows containing invalid or missing responses. In the BRFSS dataset, values like `7`, `9`, `77`, and `99` are not actual measurements; they are administrative codes indicating that a respondent answered "Don't know/Not sure" or simply refused to answer. If left untouched, the linear and logistic regression models we are implementing would treat these sentinel codes as literal mathematical weights—for example, interpreting a refused answer as 99 actual days of poor mental health—which would completely distort the gradients.

By dropping any row containing these refusal codes, we create a mathematically pure baseline dataset that we will name `x_clean_strict`. Ultimately, we cannot use this strict deletion method for our final competition pipeline, because the submission system requires us to generate predictions for every single datapoint index present in the original `x_test` file. However, establishing this strict subset serves as a perfect ablation study. It allows us to experimentally compare the performance cost of training on a larger, imputed dataset against a smaller, flawlessly clean one to see which data-preprocessing choice yields the best test error.

In [21]:
import numpy as np

# Create copies to avoid modifying the original parsed data during experimentation
x_clean = x_train_clean.copy()
y_clean = y_train.copy()
ids_clean = train_ids.copy()

# Feature index mapping based on your selected_features list
idx_menthlth = 14
idx_physhlth = 15
idx_income = 20

# 1. Map '88' (None) to '0' days for mental and physical health
x_clean[:, idx_menthlth][x_clean[:, idx_menthlth] == 88] = 0
x_clean[:, idx_physhlth][x_clean[:, idx_physhlth] == 88] = 0

# 2. Create a boolean mask of valid rows (start assuming all are valid)
valid_rows_mask = np.ones(x_clean.shape[0], dtype=bool)

# Iterate through columns to flag invalid "Refused/Don't Know" rows
for i, feature in enumerate(selected_features):
    col_data = x_clean[:, i]
    
    if feature in ['MENTHLTH', 'PHYSHLTH', 'INCOME2']:
        # For these, 77 and 99 are the refusal codes
        valid_rows_mask &= (col_data != 77) & (col_data != 99)
    elif feature == '_BMI5' or feature == '_AGEG5YR':
        # These are continuous/ordinal and don't typically use 7/9 as refusal flags in this subset
        continue
    else:
        # For all standard categorical/binary features, 7 and 9 are refusal codes
        valid_rows_mask &= (col_data != 7) & (col_data != 9)

# 3. Apply the mask symmetrically across all training arrays as a strict test
x_clean_strict = x_clean[valid_rows_mask]
y_clean_strict = y_clean[valid_rows_mask]
ids_clean_strict = ids_clean[valid_rows_mask]

print(f"Original samples: {x_train_clean.shape[0]}")
print(f"Cleaned samples (Strict):  {x_clean_strict.shape[0]}")
print(f"Samples dropped:  {x_train_clean.shape[0] - x_clean_strict.shape[0]}")

# Quick sanity check that all arrays still match
assert x_clean_strict.shape[0] == y_clean_strict.shape[0] == ids_clean_strict.shape[0]

Original samples: 328135
Cleaned samples (Strict):  222942
Samples dropped:  105193


We can see that such a strict policy deprives us of 100k+ samples, thus we consider another approach:
- **Imputation by Median/Mode**: Instead of dropping rows, we can replace refusal or "Don't know" codes with the median (for ordinal/continuous features like `INCOME2`) or the mode (most frequent answer for categorical features). This preserves our data volume while eliminating the mathematical distortion of `99` or `9` weights, finding better representations to feed our model.
- **Dedicated "Unknown" Category**: For certain categorical variables, refusing to answer might actually be a distinct demographic signal. We can map these refusal codes to a new, neutral integer (e.g., `0` or `-1`) to create an explicit "Unknown" bucket, allowing our machine learning methods to evaluate if the refusal itself is a predictor.

In [22]:
import numpy as np

class BRFSSCleaner:
    """
    Base preprocessor to clean raw BRFSS features.
    Learns parameters (like medians) from the training set to prevent data leakage,
    and applies them uniformly to any subsequent dataset (train or test).
    """
    def __init__(self, features_list):
        self.features = features_list
        # Dynamically find indices based on the provided list
        self.idx_menthlth = self.features.index('MENTHLTH')
        self.idx_physhlth = self.features.index('PHYSHLTH')
        self.idx_income = self.features.index('INCOME2')
        
        # State dictionary to remember learned statistics
        self.learned_medians_ = {}
        
    def fit(self, X):
        """Computes and stores the medians exclusively from the training data."""
        for idx in [self.idx_menthlth, self.idx_physhlth, self.idx_income]:
            col_data = X[:, idx]
            # Calculate median only on valid rows (ignoring 77 and 99)
            valid_mask = (col_data != 77) & (col_data != 99)
            
            # Save the median inside the object for later use
            self.learned_medians_[idx] = np.median(col_data[valid_mask])
            
        return self
        
    def transform(self, X):
        """Applies the cleaning and learned imputation to the provided dataset."""
        X_clean = X.copy()
        
        # 1. Map '88' to '0' days
        X_clean[:, self.idx_menthlth][X_clean[:, self.idx_menthlth] == 88] = 0
        X_clean[:, self.idx_physhlth][X_clean[:, self.idx_physhlth] == 88] = 0
        
        # 2. Impute continuous/ordinal features with the LEARNED median
        for idx in [self.idx_menthlth, self.idx_physhlth, self.idx_income]:
            col_data = X_clean[:, idx]
            missing_mask = (col_data == 77) | (col_data == 99)
            # Inject the median we calculated during fit()
            X_clean[:, idx][missing_mask] = self.learned_medians_[idx]
            
        # 3. Impute categorical/binary features with '0' (Unknown)
        for i, feature in enumerate(self.features):
            if feature in ['MENTHLTH', 'PHYSHLTH', 'INCOME2', '_BMI5', '_AGEG5YR']:
                continue
            col_data = X_clean[:, i]
            invalid_mask = (col_data == 7) | (col_data == 9)
            X_clean[:, i][invalid_mask] = 0
            
        return X_clean
        
    def fit_transform(self, X):
        """Convenience method to fit and transform the training data in one step."""
        self.fit(X)
        return self.transform(X)

# --- How to use it to safely process both sets ---
cleaner = BRFSSCleaner(selected_features)

# Fit strictly on train, and transform train
x_train_lax = cleaner.fit_transform(x_train_clean)

# Transform test using the train parameters (NO fitting here!)
x_test_lax = cleaner.transform(x_test_clean)

print(f"Processed x_train_lax: {x_train_lax.shape}")
print(f"Processed x_test_lax:  {x_test_lax.shape}")

Processed x_train_lax: (328135, 21)
Processed x_test_lax:  (109379, 21)


TODOs (either here or write class extending cleaner above in run.py as final implementation directly):
- At least 1 standardization (multiple could be beneficial ?) for continuous values (e.g. zscore or minmax)
- One Hot Encoding for categorical values (e.g. this converts the categorical values to binary vectors like for instance: [1,0,0,0] for a feature in category 1)
- Potentially append a bias to have that `w_0` in the model 

Example below:

```python
class FullPipelinePreprocessor(BRFSSCleaner):
    def fit(self, X):
        # 1. First, learn the medians using the parent class
        super().fit(X)
        
        # 2. Temporarily transform to get the imputed data
        X_temp = super().transform(X)
        
        # 3. Learn the means and standard deviations from X_temp for standardization
        # self.learned_means_ = ...
        
        return self
        
    def transform(self, X):
        # 1. Clean and impute using the parent class
        X_clean = super().transform(X)
        
        # 2. Apply One-Hot Encoding
        # 3. Apply Standardization using self.learned_means_
        # 4. np.insert(..., 1.0) for the bias term
        
        # return X_fully_processed
```